# range: private reproduction

No data, simulator, weights or results are embedded. Keep configurations, inputs and outputs outside this repository. Execution is opt-in. Read ../docs/CHAPTER_ALIGNMENT.md and ../docs/DATA_INTERFACE.md.


In [ ]:
import json
import sys
from pathlib import Path

# Install into the active kernel if needed:
# %pip install -e /path/to/radar-tracklet-thesis[neural,notebooks,test]
print("Kernel:", sys.executable)
COORDINATE = "range"
RUN_CANONICAL = False
PRIVATE_CONFIGURATION = None


## Canonical stateless GRU

25 noisy inputs predict the next clean value. GRU is the default; GRU-CNN is optional. Frozen evaluation requires original weights and scalers. New fitting requires explicit authorization and training parameters. Metrics and title-free figures go to a unique external directory. No clean teacher forcing is used.


In [ ]:
if RUN_CANONICAL:
    from radar_tracklets.experiments import external_path, run_private
    if PRIVATE_CONFIGURATION is None:
        raise ValueError("Supply an external private configuration")
    configuration = json.loads(external_path(PRIVATE_CONFIGURATION).read_text(encoding="utf-8"))
    if configuration["mode"] != "canonical" or configuration["coordinate"] != COORDINATE:
        raise ValueError("Configuration coordinate/mode does not match this notebook")
    canonical_run = run_private(configuration)
else:
    print("Canonical execution disabled.")


## Common-support comparison of archived predictions

Load external sample-level exports with trajectory identifiers and target indices. No values are entered manually. This is descriptive comparison, not test-independent selection. Plotting helpers also support named/worst trajectories. Target-time noisy-clean disturbance is a diagnostic, not the preceding input perturbation.


In [ ]:
RUN_EXPORT_COMPARISON = False
PRIVATE_PREDICTION_FILES = {}
PRIVATE_COMPARISON_DIRECTORY = None
if RUN_EXPORT_COMPARISON:
    from radar_tracklets.analysis import load_predictions, compare_common_support
    frames = {label: load_predictions(path, COORDINATE)
              for label, path in PRIVATE_PREDICTION_FILES.items()}
    common_metrics = compare_common_support(frames, PRIVATE_COMPARISON_DIRECTORY)
    print(common_metrics.to_string(index=False))


## External UKF posterior

The posterior uses a different information set from a one-step forecast. No UKF implementation or covariances are invented. Declare real/measured/filtered units separately; degree column labels are not interpreted automatically. Supply a single-case forecast export with recorded timestamps. Configure the loader using the documented API.


In [ ]:
RUN_UKF_COMPARISON = False
PRIVATE_CASE_PREDICTIONS = None
PRIVATE_UKF_CSV = None
PRIVATE_UKF_DIRECTORY = None
UKF_LOADER = None
if RUN_UKF_COMPARISON:
    from radar_tracklets.analysis import load_predictions, ukf_posterior_comparison
    if UKF_LOADER is None:
        raise ValueError("Declare UKF columns and physical units explicitly")
    ukf_metrics = ukf_posterior_comparison(
        load_predictions(PRIVATE_CASE_PREDICTIONS, COORDINATE), PRIVATE_UKF_CSV,
        PRIVATE_UKF_DIRECTORY, **UKF_LOADER)
    print(ukf_metrics.to_string(index=False))
